# Code to reproduce the results in the technical appendix

In [ ]:
import pandas as pd
import ast 
import altair as alt

from discovery_child_development import PROJECT_DIR, S3_BUCKET, logging
from discovery_child_development.analysis.initial_results import utils
from nesta_ds_utils.loading_saving import S3

from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu
from discovery_child_development.utils import chart_trends

# Remove altair warning
import altair as alt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

S3_OUTPUTS_DIR = '2024-07-iss-child-development/outputs/'

In [ ]:
TECH = 'Technology'

# Taxonomy dataframe
topics_df = utils.load_topic_data()

# List all tech major categories
tech_subtypes = set(topics_df.query("type == @TECH").subtype.unique())
print(tech_subtypes)

## Helper functions

In [ ]:
# Relevant document ids for time series
def get_tech_ids(data_exploded_df):
    """Get relevant document ids for time series/growth estimations"""
    return (
        data_exploded_df
        .query("type == @TECH")
        .query("year >= 2013")
        .drop_duplicates('id')
        .id.to_list()
)

def get_tech_ids_5y(data_exploded_df: pd.DataFrame) -> pd.DataFrame:
    """Get relevant document ids for 2019-2023 stats"""
    return (
        data_exploded_df
        .query("type == @TECH")
        .query("year >= 2019")
        .drop_duplicates('id')
        .id.to_list()
    )

def report_magnitude_growth(magnitude_growth_df: pd.DataFrame) -> None:
    """Report magnitude and growth"""
    # Smoothed growth in 2019-2023
    growth = magnitude_growth_df.growth.iloc[0]
    # Total funding in 2019-2023 (in millions)
    magnitude = magnitude_growth_df.magnitude.iloc[0] * 5 

    logging.info(f"Growth in 2019-2023: {growth:.2f}%")
    logging.info(f"Total in 2019-2023: {magnitude:.2f}")

    return growth, magnitude


def get_tech_distribution(data_exploded_df: pd.DataFrame, tech_ids_5y, values: list, column: str='subtype') -> pd.DataFrame:
    """Get distribution of technology projects"""
    return utils.get_data_distribution(
        (
            data_exploded_df
            .query('id in @tech_ids_5y')
            .query("type == @TECH")
            .drop_duplicates(['id', column])
        ),
        column=column, 
        values=values,
    ) 


def get_tech_ts(data_exploded_df: pd.DataFrame, tech_ids, values: list, column: str='subtype') -> pd.DataFrame:
    """Get distribution of technology projects"""
    return utils.get_data_distribution(
        (
            data_exploded_df
            .query('id in @tech_ids')
            .query("type == @TECH")
            .drop_duplicates(['id', column])
        ),
        column=column, 
        values=values,
        ts = True,
    )  

def get_tech_trends(data_exploded_df: pd.DataFrame, tech_ids, tech_ids_5y, values: list, column='subtype') -> pd.DataFrame:
    tech_dist = get_tech_distribution(data_exploded_df, tech_ids_5y, values, column=column)
    tech_ts = get_tech_ts(data_exploded_df, tech_ids, values, column=column)
    _value = 'counts' if values[0] == 'id' else values[0]
    tech_magnitude_growth = utils.magnitude_and_growth(
        tech_ts, 
        column = column, 
        value=_value)
    tech_trends = (
        tech_dist.merge(
            tech_magnitude_growth, 
            on=column, 
            how='left')
    )
    return tech_trends

def get_application_trends(data_exploded_df: pd.DataFrame, tech_ids, tech_ids_5y, values: list, column:str = 'type', hide_categories: list = ['Technology', 'General']) -> pd.DataFrame:
    """Get application trends"""
    tech_applications_df = utils.get_data_distribution(
        data_exploded_df.query('id in @tech_ids_5y'), 
        column=column, 
        values=values,
    ) 
    trends_df = utils.get_data_magnitude_growth(
        data_exploded_df, ids=tech_ids, 
        column=column, 
        value=values[0])  
    return (
        tech_applications_df
        .merge(trends_df.drop('counts', axis=1), on=column, suffixes=('', '_'))
        .query(f"{column} not in @hide_categories")
        ) 


def trends_chart(application_trends_df: pd.DataFrame) -> pd.DataFrame:
    return (
        alt.Chart(application_trends_df)
        .mark_point()
        .encode(
            x='magnitude:Q',
            y='growth:Q',
            color='type:N',
            tooltip=['type', 'magnitude', 'growth'],
        )
    )

## Research funding
- Fig 2: Growth and total early-years digital tech research funding in 2019-2023
- Breakdowns by funder type (top funder; proportion from Innovate UK)
- Fig 4: Proportion of early-years funding associated with digital technologies
- Fig 5: Proportion and growth of major digital tech categories in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline funding growth across all sectors in 2019-2023


In [ ]:
# Load the data
ukri_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_ukri_2024.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
ukri_exploded_df = utils.explode_data(ukri_df).query("topics != 'arts'")

# Get relevant document ids
ukri_tech_ids = get_tech_ids(ukri_exploded_df)
ukri_tech_ids_5y = get_tech_ids_5y(ukri_exploded_df)


In [ ]:
logging.info(f"Total number of UK research projects {len(ukri_df)}")

### Growth and total early-years digital tech research funding

Figure 2

In [ ]:
# Filter only technology projects
ukri_tech_type_df = (
    ukri_exploded_df
    .query("id in @ukri_tech_ids")
    .drop_duplicates(['id'])
)

ukri_ts_amounts_tech = utils.get_timeseries(ukri_tech_type_df, column='amount')
ukri_ts_counts_tech = utils.get_timeseries(ukri_tech_type_df, column='id')
utils.plot_quick_ts(ukri_ts_amounts_tech, 'amount')

In [ ]:
# Get magnitude and growth
magnitude_growth_ukri = au.ts_magnitude_growth_(
    ukri_ts_amounts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_ukri)

### Breakdowns by funder type
- Top funders
- Proportion from Innovate UK

In [ ]:
# Check who are the top research funders for early-years digital tech
top_funders = (
    ukri_exploded_df
    .drop_duplicates(['id', 'lead_funder'])
    .query("id in @ukri_tech_ids_5y")
    .query("year <= 2023")    
    .groupby('lead_funder')
    .agg(amount=('amount', 'sum'))
    .sort_values('amount', ascending=False)
    .assign(proportion = lambda df: df.amount / df.amount.sum())
)
top_funders.head(5)

In [ ]:
# Projects funded in the past five years
_funding_df = (
    ukri_exploded_df
    .query("id in @ukri_tech_ids_5y")
    .query("year <= 2023") 
    .drop_duplicates('id')
)

# Get the total funding
funding_total = _funding_df.amount.sum()

# Get Innovate UK funding specifically
funding_innovate_uk = (
    _funding_df
    .query("lead_funder == 'Innovate UK'")
    .amount.sum())

# Calculate proportion of funding by Innovate UK
proportion_innovate_uk = funding_innovate_uk / funding_total

logging.info(f"Propotion of funding by Innovate UK: {proportion_innovate_uk:.2f}")

In [ ]:
# Get funding excluding health-related projects (for reference)
health_ids = ukri_exploded_df.query("type == 'Health'").id.to_list()
funding_wout_health = _funding_df.query("id not in @health_ids").amount.sum()

# Calculate proportion of funding by Innovate UK excluding health
funding_innovate_uk_wout_health = (
    _funding_df
    .query("lead_funder == 'Innovate UK'")
    .query("id not in @health_ids")
    .amount.sum())

proportion_innovate_uk_wout_health = funding_innovate_uk_wout_health / funding_wout_health

logging.info(f"Propotion of funding by Innovate UK excluding health: {proportion_innovate_uk_wout_health:.2f}")

### Proportion of early-years funding associated with digital technologies

Figure 4

In [ ]:
# Calculate the total early-years project funding
total_early_years_funding = (
    ukri_exploded_df
    .query("year >= 2019 and year <= 2023")
    .drop_duplicates('id')
    .amount.sum())

# Get the proportion of funding for early-years digital tech
proportion_early_years_tech_ukri = funding_total / total_early_years_funding
logging.info(f"Proportion of funding for early-years digital tech: {proportion_early_years_tech_ukri:.2f}")

In [ ]:
# Check Technology proportion using another approach
utils.get_data_distribution(
    ukri_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id', 'amount']
)

### Digital tech trends in 2019-2023 (funding)

Figure 5

In [ ]:
ukri_tech_trends = get_tech_trends(
    ukri_exploded_df.query("year <= 2023") , 
    ukri_tech_ids, 
    ukri_tech_ids_5y, 
    ['amount', 'id']
)

ukri_tech_trends

### Application area trends in 2019-2023 (funding)

Figure 6

In [ ]:
ukri_application_trends = get_application_trends(
    ukri_exploded_df.query("year <= 2023") , 
    ukri_tech_ids, 
    ukri_tech_ids_5y, 
    ['amount', 'id']
)

ukri_application_trends


Trends information for the heat map.

Note that the trend typology is data-informed - we're guided by the results below - but we might also slighty adjust the final trends category in a few select cases.

In [ ]:
chart_trends.estimate_trend_type(
    ukri_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]

Designating Dev and learning as stabilising as the magnitue is in fact quite large relatively speaking

In [ ]:
fig = trends_chart(
    ukri_application_trends
    .query("type != 'Biosciences'")
)
fig

### Baseline funding growth across all sectors

In [ ]:
gtr_df = S3.download_obj(
    bucket = S3_BUCKET,
    path_from = S3_OUTPUTS_DIR + 'gtr_texts.csv',
    download_as='dataframe'
)

In [ ]:
ukri_baseline_df = utils.get_baseline_ukri(gtr_df)

trends_baseline = au.ts_magnitude_growth_(
    ts_df = ukri_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
ukri_baseline_magnitude = trends_baseline.loc['amount'].magnitude
ukri_baseline_growth = trends_baseline.loc['amount'].growth
logging.info(f"UKRI baseline growth: {ukri_baseline_growth:.2f}%")

In [ ]:
_gtr_all_projects_df = (
    gtr_df
    .assign(year = lambda df: df.start.apply(lambda x: int(x[0:4])))
    .query("year >= 2019 and year <= 2023")
)

In [ ]:
ukri_all_projects_funding = _gtr_all_projects_df.dropna(subset=["leadFunder"]).amount.sum()
ukri_innovate_uk_funding = _gtr_all_projects_df.query("leadFunder == 'Innovate UK'").amount.sum()
prop_innovate_uk = (ukri_innovate_uk_funding / ukri_all_projects_funding)

logging.info(f"Proportion of Innovate UK funding: {prop_innovate_uk:.2f}")

## Research publications

- Fig 2: Growth and total early-years digital tech publications in 2019-2023
- Fig 4: Proportion of early-years publications associated with digital technologies
- Fig 5: Proportion and growth of major digital tech categories in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline publication growth across all sectors in 2019-2023
- Geographical distribution


In [ ]:
# Load the data
openalex_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_openalex_2024.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
openalex_exploded_df = utils.explode_data(openalex_df).query("topics != 'arts'")

# Get relevant document ids
openalex_tech_ids = get_tech_ids(openalex_exploded_df)
openalex_tech_ids_5y = get_tech_ids_5y(openalex_exploded_df)


In [ ]:
logging.info(f"Total number of publications {len(openalex_df)}")

### Growth and total early-years digital tech publications

Figure 2

In [ ]:
# Filter only technology projects
openalex_tech_type_df = (
    openalex_exploded_df
    .query("id in @openalex_tech_ids")
    .drop_duplicates(['id'])
)

openalex_ts_counts_tech = utils.get_timeseries(openalex_tech_type_df, column='id')
utils.plot_quick_ts(openalex_ts_counts_tech.query("year >= 2017"), 'counts')

In [ ]:
# Get magnitude and growth
magnitude_growth_openalex = au.ts_magnitude_growth_(
    openalex_ts_counts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_openalex)

### Proportion of early-years publications associated with digital technologies

Figure 4

In [ ]:
# Check Technology proportion
openalex_category_distribution_df = utils.get_data_distribution(
    openalex_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id']
)

# Get the proportion of technology projects
openalex_prop = openalex_category_distribution_df.query("type == @TECH").counts_prop.iloc[0]
logging.info(f"Proportion of publications: {openalex_prop:.2f}")

# See all major categories
openalex_category_distribution_df

### Digital tech trends in 2019-2023 (publications)

Figure 5

In [ ]:
openalex_tech_trends = get_tech_trends(
    openalex_exploded_df.query("year <= 2023") , 
    openalex_tech_ids, 
    openalex_tech_ids_5y, 
    ['id']
)

openalex_tech_trends

### Application area trends in 2019-2023 (publications)

Figure 6

In [ ]:
openalex_application_trends = get_application_trends(
    openalex_exploded_df.query("year <= 2023"), 
    openalex_tech_ids, 
    openalex_tech_ids_5y, 
    ['id']
)

openalex_application_trends


Trends information for the heat map

In [ ]:
chart_trends.estimate_trend_type(
    openalex_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)

In [ ]:
fig = trends_chart(openalex_application_trends)
fig

### Geographical distribution

In [ ]:
_openalex_countries_df = (
    openalex_exploded_df
    .query('subtype in @tech_subtypes')
    .query("year <= 2023") 
    .assign(country_code = lambda df: df.country_code.apply(ast.literal_eval))
    .explode('country_code')
    .drop_duplicates(['id', 'country_code'])
)

n_total_with_codes = len(
    _openalex_countries_df
    .drop_duplicates('id')
    .query("year >= 2019")
    .query("year <= 2023") 
    .dropna(subset=['country_code'])
)

openalex_countries_df, _ = utils.get_geographical_distribution(_openalex_countries_df)
openalex_countries_df = (
    openalex_countries_df
    .assign(total = lambda df: df.magnitude*5)
    .assign(proportion = lambda df: df.total / n_total_with_codes)
    .sort_values('proportion', ascending=False)
)
openalex_countries_df.head(10)

In [ ]:
openalex_US = openalex_countries_df.query("country_code=='US'").proportion.iloc[0]
logging.info(f'US proportion: {openalex_US:.2f}')

openalex_GB = openalex_countries_df.query("country_code=='GB'").proportion.iloc[0]
logging.info(f'GB proportion: {openalex_GB:.2f}')

### Baseline growth across all sectors (publications)

In [ ]:
# Load the data
openalex_baseline_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_openalex.csv',
        download_as='dataframe'
    )
)

In [ ]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = openalex_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
openalex_baseline_magnitude = trends_baseline.loc['counts'].magnitude
openalex_baseline_growth = trends_baseline.loc['counts'].growth
logging.info(f"Publication baseline growth: {openalex_baseline_growth:.2f}%")

### Baseline growth for edtech (publications)

In [ ]:
# Load the data
openalex_baseline_df_edtech = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_openalex_edtech.csv',
        download_as='dataframe'
    )
)

trends_baseline = au.ts_magnitude_growth_(
    ts_df = openalex_baseline_df_edtech,
    year_start = 2019,
    year_end = 2023  
)
openalex_baseline_magnitude_edtech = trends_baseline.loc['counts'].magnitude*5
openalex_baseline_growth_edtech = trends_baseline.loc['counts'].growth
logging.info(f"Edtech publications in total: {openalex_baseline_magnitude_edtech:.2f}")
logging.info(f"Edtech publication baseline growth: {openalex_baseline_growth_edtech:.2f}%")

## Patents

- Fig 2: Growth and total early-years digital tech patents in 2019-2023
- Fig 4: Proportion of early-years patents associated with digital technologies
- Fig 5: Proportion and growth of major digital tech categories in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline patent growth across all sectors in 2019-2023
- Geographical distribution


In [ ]:
# Load the data
patents_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_patents_2024.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
patents_exploded_df = utils.explode_data(patents_df).query("topics != 'arts'")

# Get relevant document ids
patents_tech_ids = get_tech_ids(patents_exploded_df)
patents_tech_ids_5y = get_tech_ids_5y(patents_exploded_df)

### Growth and total early-years digital tech patents

Figure 2

In [ ]:
# Filter only technology projects
patents_tech_type_df = (
    patents_exploded_df
    .query("id in @patents_tech_ids")
    .drop_duplicates(['id'])
)

patents_ts_counts_tech = utils.get_timeseries(patents_tech_type_df, column='id')
utils.plot_quick_ts(patents_ts_counts_tech, 'counts')

In [ ]:
# Get magnitude and growth
magnitude_growth_patents= au.ts_magnitude_growth_(
    patents_ts_counts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_patents)

Check stats without including China

In [ ]:
# Filter only technology projects
patents_tech_type_df_wout_CN = (
    patents_exploded_df
    .query('country_code != "CN"')
    .query("id in @patents_tech_ids")
    .drop_duplicates(['id'])
)

ts_counts_tech_wout_CN = utils.get_timeseries(patents_tech_type_df_wout_CN, column='id')
utils.plot_quick_ts(ts_counts_tech_wout_CN, 'counts')

In [ ]:
# Get magnitude and growth
magnitude_growth_patents_wout_CN = au.ts_magnitude_growth_(
    ts_counts_tech_wout_CN,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_patents_wout_CN)

### Proportion of early-years patents associated with digital technologies

Figure 4

In [ ]:
# Check Technology proportion
patents_category_distribution_df = utils.get_data_distribution(
    patents_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id']
)

# Get the proportion of technology projects
patents_prop = patents_category_distribution_df.query("type == @TECH").counts_prop.iloc[0]
logging.info(f"Proportion of technology patents: {patents_prop:.2f}")

# See all major categories
patents_category_distribution_df

### Digital tech trends in 2019-2023 (patents)

Figure 5

In [ ]:
patents_tech_trends = get_tech_trends(
    patents_exploded_df.query("year <= 2023"), 
    patents_tech_ids, 
    patents_tech_ids_5y, 
    ['id']
)

patents_tech_trends

### Application area trends in 2019-2023 (patents)

Figure 6

In [ ]:
patent_application_trends = get_application_trends(
    patents_exploded_df.query("year <= 2023"), 
    patents_tech_ids, 
    patents_tech_ids_5y, 
    ['id']
)

patent_application_trends

Trends information for the heat map

In [ ]:
chart_trends.estimate_trend_type(
    patent_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)

- Society has such a tiny number of patents we'll designate as 'dormant' instead
- Development and learning had a growth rate very close to 0, so not that strongly emerging - more between dormant and emerging.

In [ ]:
fig = trends_chart(patent_application_trends)
fig

### Geographical distribution

In [ ]:
_patents_countries_df = (
    patents_exploded_df
    .query("year <= 2023")
    .query('subtype in @tech_subtypes')
)

n_total_with_codes = len(
    _patents_countries_df
    .drop_duplicates('id')
    .query("year >= 2019")
    .query("year <= 2023")
    .dropna(subset=['country_code'])
)

patents_countries_df, _ = utils.get_geographical_distribution(_patents_countries_df)
patents_countries_df = (
    patents_countries_df
    .assign(proportion = lambda df: df.magnitude*5 / n_total_with_codes)
    .sort_values('proportion', ascending=False)
)
patents_countries_df.head(15)

In [ ]:
openalex_US = patents_countries_df.query("country_code=='CN'").proportion.iloc[0]
logging.info(f'CN proportion: {openalex_US:.2f}')

openalex_US = patents_countries_df.query("country_code=='US'").proportion.iloc[0]
logging.info(f'US proportion: {openalex_US:.2f}')

openalex_GB = patents_countries_df.query("country_code=='GB'").proportion.iloc[0]
logging.info(f'GB proportion: {openalex_GB:.2f}')

### Baseline growth across all sectors 

In [ ]:
# Load the data
patents_baseline_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_patents.csv',
        download_as='dataframe'
    )
)

In [ ]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = patents_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
patents_baseline_magnitude = trends_baseline.loc['counts'].magnitude
patents_baseline_growth = trends_baseline.loc['counts'].growth
logging.info(f"UKRI baseline growth: {patents_baseline_growth:.2f}%")

## Venture funding

- Fig 2: Growth and total early-years digital tech funding in 2019-2023
- Fig 3: Venture funding by deal size over years
- Fig 4: Proportion of early-years funding associated with digital technologies
- Fig 5: Proportion and growth of major digital tech funding in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline funding growth across all sectors in 2019-2023
- Geographical distribution


In [ ]:
# Load the data
crunchbase_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_crunchbase_2024.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
crunchbase_exploded_df = utils.explode_data(crunchbase_df, is_crunchbase=True).query("topics != 'arts'")

# Get relevant document ids
crunchbase_tech_ids = get_tech_ids(crunchbase_exploded_df)
crunchbase_tech_ids_5y = get_tech_ids_5y(crunchbase_exploded_df)


In [ ]:
logging.info(f"Total number of venture funding rounds {len(crunchbase_df)}")

### Growth and total early-years digital tech investment

Figure 2

In [ ]:
# Filter only technology projects
crunchbase_tech_type_df = (
    crunchbase_exploded_df
    .query("id in @crunchbase_tech_ids")
    .drop_duplicates(['id'])
)

crunchbase_ts_amounts_tech = utils.get_timeseries(crunchbase_tech_type_df, column='amount')
crunchbase_ts_counts_tech = utils.get_timeseries(crunchbase_tech_type_df, column='id')
utils.plot_quick_ts(crunchbase_ts_amounts_tech, 'amount')

In [ ]:
# Get magnitude and growth
magnitude_growth_crunchbase = au.ts_magnitude_growth_(
    crunchbase_ts_amounts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_crunchbase)

Check growth if removing large deals (larger than £100M)

In [ ]:
# Filter only technology projects
crunchbase_tech_type_df_wout_large = (
    crunchbase_exploded_df
    .query("amount < 100000")
    .query("id in @crunchbase_tech_ids")
    .drop_duplicates(['id'])
)

ts_amounts_tech = utils.get_timeseries(crunchbase_tech_type_df_wout_large, column='amount')
ts_counts_tech = utils.get_timeseries(crunchbase_tech_type_df_wout_large, column='id')
utils.plot_quick_ts(ts_amounts_tech, 'amount')

In [ ]:
# Get magnitude and growth
magnitude_growth_crunchbase = au.ts_magnitude_growth_(
    ts_amounts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_crunchbase)

### Venture funding by deal size over years

Figure 3

In [ ]:
deal_order = ["n/a", "£0-5M", "£5-20M", "£20-100M", "£100M+"]

def deal_amount_to_range_coarse(
    amount: float, currency: str = "£", categories: bool = True
) -> str:
    """
    Convert amounts to range in millions
    Args:
        amount: Investment amount (in GBP thousands)
        categories: If True, adding indicative deal categories
        currency: Currency symbol
    """
    amount /= 1e3
    if (amount >= 0.001) and (amount <= 5):
        return f"{currency}0-5M" if not categories else f"{currency}0-5M"
    elif (amount > 5) and (amount <= 20):
        return f"{currency}5-20M" if not categories else f"{currency}5-20M"
    elif (amount > 20) and (amount <= 100):
        return f"{currency}20-100M" if not categories else f"{currency}20-100M"
    elif amount > 100:
        return f"{currency}100M+"
    else:
        return "n/a"

In [ ]:
funding_df_ranges = (
    crunchbase_df
    .query("id in @crunchbase_tech_ids")
    .assign(_amount = lambda df: df.amount/1000)
    .assign(deal_type=lambda df: df.amount.apply(deal_amount_to_range_coarse))
    .astype({"deal_type": "category"})
    .assign(deal_type=lambda x: x.deal_type.cat.set_categories(deal_order))
    .drop_duplicates('id')
)

deal_data = (
    funding_df_ranges.groupby(["year", "deal_type"], as_index=True)
    .agg(
        counts=("id", "count"),
        total_amount=("amount", "sum"),
    )
    .reset_index()
    .query("year >= 2013")
    .query("year <= 2023")
    .assign(total_amount=lambda df: df.total_amount / 1000)
)
deal_data_wide_df = (
    deal_data.pivot(index="year", columns="deal_type", values="total_amount")
    .fillna(0)
    .astype(int)
    .reset_index()
)

deal_data_wide_df = (
    deal_data_wide_df
    .merge(deal_data.groupby('year').total_amount.sum().reset_index(), on='year', how='left')
)

deal_data_wide_df

In [ ]:
deal_data_wide_counts_df = (
    deal_data.pivot(index="year", columns="deal_type", values="counts")
    .fillna(0)
    .astype(int)
    .reset_index()
)
deal_data_wide_counts_df

In [ ]:
# Sense check
deal_data_wide_df.query("year >= 2019 and year <= 2023").total_amount.sum()

### Proportion of early-years investment associated with digital technologies

Figure 4

In [ ]:
# Check Technology proportion using another approach
crunchbase_category_distribution_df = utils.get_data_distribution(
    crunchbase_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id', 'amount']
)

# Get the proportion of technology projects
crunchbase_prop = crunchbase_category_distribution_df.query("type == @TECH").amount_prop.iloc[0]
logging.info(f"Proportion of investment: {crunchbase_prop:.2f}")

# See all major categories
crunchbase_category_distribution_df

### Digital tech trends in 2019-2023 (investment)

Figure 5

In [ ]:
crunchbase_tech_trends = get_tech_trends(
    crunchbase_exploded_df.query("year <= 2023"), 
    crunchbase_tech_ids, 
    crunchbase_tech_ids_5y, 
    ['amount', 'id']
)

crunchbase_tech_trends

### Application area trends in 2019-2023 (investment)

Figure 6

In [ ]:
crunchbase_application_trends = get_application_trends(
    crunchbase_exploded_df.query("year <= 2023"), 
    crunchbase_tech_ids, 
    crunchbase_tech_ids_5y, 
    ['amount', 'id']
)

crunchbase_application_trends


Trends information for the heat map

In [ ]:
chart_trends.estimate_trend_type(
    crunchbase_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]

In [ ]:
fig = trends_chart(
    crunchbase_application_trends
    .query("type != 'Biosciences'")
)
fig

### Geographical distribution

In [ ]:
_crunchbase_countries_df = (
    crunchbase_exploded_df
    .query("year <= 2023")
    .query('id in @crunchbase_tech_ids')
)

n_total_with_codes = (
    _crunchbase_countries_df
    .query("year >= 2019")
    .query("year <= 2023")
    .drop_duplicates('id')
    .dropna(subset=['country_code'])
    .amount.sum()
)

crunchbase_countries_df, _ = utils.get_geographical_distribution(
    _crunchbase_countries_df,
    column = 'amount',
)
crunchbase_countries_df = (
    crunchbase_countries_df
    .assign(total = lambda df: df.magnitude*5)
    .assign(proportion = lambda df: df.total / n_total_with_codes)
    .sort_values('proportion', ascending=False)
)
crunchbase_countries_df.head(10)

In [ ]:
# Double check GBR total investment
(
    crunchbase_exploded_df
    .query("id in @crunchbase_tech_ids_5y")
    .query("year <= 2023")
    .query("country_code == 'GBR'")
    .drop_duplicates('id')
    .amount.sum()
)

In [ ]:
# Double check investment types
sorted(list(crunchbase_exploded_df.investment_type.unique()))

### Baseline growth across all sectors (venture funding)

In [ ]:
# Load the data
crunchbase_baseline_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_crunchbase.csv',
        download_as='dataframe'
    )
)

In [ ]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = crunchbase_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
crunchbase_baseline_magnitude = trends_baseline.loc['amount'].magnitude
crunchbase_baseline_growth = trends_baseline.loc['amount'].growth
logging.info(f"Investment baseline growth: {crunchbase_baseline_growth:.2f}%")

### Baseline growth across all sectors (edtech)

In [ ]:
# Load the data
crunchbase_baseline_df_edtech = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_crunchbase_edtech.csv',
        download_as='dataframe'
    )
)

trends_baseline = au.ts_magnitude_growth_(
    ts_df = crunchbase_baseline_df_edtech,
    year_start = 2019,
    year_end = 2023  
)
crunchbase_baseline_magnitude_edtech = trends_baseline.loc['amount'].magnitude*5
crunchbase_baseline_growth_edtech = trends_baseline.loc['amount'].growth
logging.info(f"Edtech investment baseline growth: {crunchbase_baseline_growth_edtech:.2f}%")
logging.info(f"Edtech magnitude: {crunchbase_baseline_magnitude_edtech:.2f}")

# Technical appendix

In [ ]:
FIGURE_DIR = PROJECT_DIR / 'outputs/figures/final'

CATS = [
    'Health',
    'Development & learning',
    'Child care & preschool',
    'Parenting',
    'Society',
    'Biosciences',
]

TS_FIG_HEIGHT = 150
TS_FIG_WIDTH = 250
DEF_COLOUR = pu.NESTA_COLOURS[0]
SECONDARY_COLOUR = pu.NESTA_COLOURS[1]

## Detailed time series for Figure 2

### Data used for growth estimates

In [ ]:
def simple_bar_chart(
        data,
        y_field,
        title,
        denominator=1,
        colour=DEF_COLOUR,
        round_int=1,):
    fig = (alt.Chart(
            (
                data
                .assign(**{y_field: lambda df: df[y_field] / denominator})
            ),
            width = 350,
            height = 150,
        ).encode(
            x=alt.X('year:O', title=''),
            y=alt.Y(f'{y_field}:Q', title=''),
            color=alt.value((pu.NESTA_COLOURS[0])),
            tooltip=['year', y_field],
        )
    )
    
    fig_text = (
        fig
        .encode(
            text=alt.Text(f'{y_field}:Q', format=f'.{round_int}f')
        )
    )
    fig = fig.mark_bar().properties(title=title) + fig_text.mark_text(dy=-5)
    return (
        pu.configure_plots(fig, title)
        .configure_legend(title=None)
    )
    # return fig


In [ ]:
fig = simple_bar_chart(
    ukri_ts_amounts_tech,
    'amount',
    'Research funding (£ millions)',
    1e+3)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_ukri_digital_tech.png', scale_factor=2.0)

In [ ]:
fig = simple_bar_chart(
    openalex_ts_counts_tech.query("year >= 2017"),
    'counts',
    'Research publication counts',
    1,
    round_int=0)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_openalex_digital_tech.png', scale_factor=2.0)

In [ ]:
fig = simple_bar_chart(
    patents_ts_counts_tech,
    'counts',
    'Patent application counts',
    1,
    round_int=0)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_patents_digital_tech.png', scale_factor=2.0)

In [ ]:
fig = simple_bar_chart(
    ts_counts_tech_wout_CN,
    'counts',
    'Patent application counts (excluding China)',
    1,
    round_int=0)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_patents_wout_china_digital_tech.png', scale_factor=2.0)

In [ ]:
fig = simple_bar_chart(
    crunchbase_ts_amounts_tech,
    'amount',
    'Venture funding (£ millions)',
    1e+3)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_crunchbase_digital_tech.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Figure2_crunchbase_digital_tech.html')

In [ ]:
fig = simple_bar_chart(
    crunchbase_ts_amounts_tech,
    'amount',
    'Venture funding (£ millions)',
    1e+3)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_crunchbase_digital_tech.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Figure2_crunchbase_digital_tech.html')

### Baselines

In [ ]:
fig = simple_bar_chart(
    ukri_baseline_df,
    'amount',
    'Baseline research funding (£ billions)',
    1e+6,
    round_int=2)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_ukri_baseline.png', scale_factor=2.0)

In [ ]:
fig = simple_bar_chart(
    openalex_baseline_df.query("year >= 2017"),
    'counts',
    'Baseline research publication counts (millions)',
    1e+6,
    round_int=2)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_openalex_baseline.png', scale_factor=2.0)

In [ ]:
fig = simple_bar_chart(
    patents_baseline_df,
    'counts',
    'Baseline patent application counts (millions)',
    1e+6,
    round_int=2)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_patents_baseline.png', scale_factor=2.0)

In [ ]:
fig = simple_bar_chart(
    crunchbase_baseline_df,
    'amount',
    'Baseline venture funding (£ billions)',
    1e+6)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_crunchbase_baseline.png', scale_factor=2.0)

## Digital tech

### Detailed time series, all datasets

In [ ]:
def create_bar_chart(data, dataset_name, y_field, y_title, category, colour=DEF_COLOUR):
    return (
        alt.Chart(
            (
                data
                .query(f"dataset == '{dataset_name}'")
                .query(f"type == '{category}'")
            ),
            height=TS_FIG_HEIGHT,
            width=TS_FIG_WIDTH,
        )
        .mark_bar()
        .encode(
            x=alt.X('year:O', title=''),
            y=alt.Y(f'{y_field}:Q', title=y_title),
            color=alt.value(colour),
            tooltip=['year', y_field],
        )
        .properties(
            title=f'{category} {dataset_name.lower()}'
        )
    )

def create_step_chart(data, dataset_name, y_field, y_title, category, colour=SECONDARY_COLOUR):
    return (
        alt.Chart(
            (
                data
                .query(f"dataset == '{dataset_name}'")
                .query(f"type == '{category}'")
            ),
            height=TS_FIG_HEIGHT,
            width=TS_FIG_WIDTH,
        )
        .mark_line(interpolate='step')
        .encode(
            x=alt.X('year:O', title=''),
            y=alt.Y(f'{y_field}:Q', title=y_title),
            color=alt.value(colour),
            tooltip=['year', y_field],
        )
    )

def category_ts_figs(category_ts, category):
    ukri_step_fig = create_step_chart(
        category_ts, 
        'Research funding', 
        'counts', 
        'Project counts', 
        category
    )

    ukri_funds_fig = create_bar_chart(
        category_ts, 
        'Research funding', 
        'amount', 
        'Research funding (£ millions)', 
        category
    )

    ukri_ts_fig = alt.layer(ukri_funds_fig, ukri_step_fig).resolve_scale(y='independent')

    openalex_ts_fig = create_bar_chart(
        category_ts, 
        'Publications', 
        'counts', 
        'Publication counts', 
        category
    )

    patents_ts_fig = create_bar_chart(
        category_ts, 
        'Patents', 
        'counts', 
        'Patent counts', 
        category
    )

    crunchbase_step_fig = create_step_chart(
        category_ts, 
        'Venture funding', 
        'counts', 
        'Funding round counts', 
        category
    )

    crunchbase_funds_fig = create_bar_chart(
        category_ts, 
        'Venture funding', 
        'amount', 
        'Venture funding (£ millions)', 
        category
    )

    crunchbase_ts_fig = alt.layer(crunchbase_funds_fig, crunchbase_step_fig).resolve_scale(y='independent')

    upstream_combined_fig = alt.hconcat(
        ukri_ts_fig,
        openalex_ts_fig,
        spacing = 20,
    )

    downstream_combined_fig = alt.hconcat(
        patents_ts_fig,
        crunchbase_ts_fig,
        spacing = 60,
    )

    combined_fig = alt.vconcat(
        upstream_combined_fig,
        downstream_combined_fig,
        spacing = 30,
    )

    return pu.configure_plots(combined_fig)

In [ ]:
ukri_tech_ts = (
    get_tech_ts(
        ukri_exploded_df,
        ukri_tech_ids,
        ['id', 'amount'],
    )
    .assign(dataset = 'Research funding')
)

openalex_tech_ts = (
    get_tech_ts(
        openalex_exploded_df,
        openalex_tech_ids,
        ['id'],
    )
    .assign(dataset = 'Publications')
)
# Replace counts values with 0 for all columns 'years' betwene 2013 and 2016
openalex_tech_ts.loc[
    openalex_tech_ts.year < 2017, 
    openalex_tech_ts.columns.str.contains('counts')
] = 0

patents_tech_ts = (
    get_tech_ts(
        patents_exploded_df,
        patents_tech_ids,
        ['id'],
    )
    .assign(dataset = 'Patents')
)

crunchbase_tech_ts = (
    get_tech_ts(
        crunchbase_exploded_df,
        crunchbase_tech_ids,
        ['id', 'amount'],
    )
    .assign(dataset = 'Venture funding')
)

tech_ts_df = (
    pd.concat([ukri_tech_ts, openalex_tech_ts, patents_tech_ts, crunchbase_tech_ts])
    .assign(amount = lambda df: df.amount/1e3)
    .drop('type', axis=1)
    .rename(columns = {'subtype': 'type'})
)


In [ ]:
for cat in ['AI', 'Mobile', 'Internet', 'Immersive tech']:
    fig = category_ts_figs(tech_ts_df.query("year <= 2023"), category=cat)
    # save as png with dpi=300
    fig.save(FIGURE_DIR / f'Digital_tech_{cat}_ts.png', scale_factor=2.0)
    # save as html
    fig.save(FIGURE_DIR / f'Digital_tech_{cat}_ts.html')


## Application areas

### Detailed time series, all datasets

In [ ]:
# Prepare time series data for all applications
ukri_applications_ts = (
    utils.get_data_distribution(
        ukri_exploded_df.query('id in @ukri_tech_ids'),
        column='type', 
        values=['id', 'amount'],
        ts=True
    )
    .assign(dataset = 'Research funding')
)

openalex_applications_ts = (
    utils.get_data_distribution(
        openalex_exploded_df.query('id in @openalex_tech_ids'),
        column='type', 
        values=['id'],
        ts=True
    )
    .assign(dataset = 'Publications')
)
# Replace counts values with 0 for all columns 'years' betwene 2013 and 2016
openalex_applications_ts.loc[
    openalex_applications_ts.year < 2017, 
    openalex_applications_ts.columns.str.contains('counts')
] = 0

patents_applications_ts = (
    utils.get_data_distribution(
        patents_exploded_df.query('id in @patents_tech_ids'),
        column='type', 
        values=['id'],
        ts=True
    )
    .assign(dataset = 'Patents')
)

crunchbase_applications_ts = (
    utils.get_data_distribution(
        crunchbase_exploded_df.query('id in @crunchbase_tech_ids'),
        column='type', 
        values=['id', 'amount'],
        ts=True
    )
    .assign(dataset = 'Venture funding')
)

applications_ts = (
    pd.concat([
        ukri_applications_ts,
        openalex_applications_ts,
        patents_applications_ts,
        crunchbase_applications_ts
    ], ignore_index=True)
    # Convert to millions
    .assign(amount = lambda df: df.amount / 1000)
)

In [ ]:
for cat in CATS:
    fig = category_ts_figs(applications_ts.query("year <= 2023"), cat)
    # save as png with dpi=300
    fig.save(FIGURE_DIR / f'Application_areas_{cat}_ts.png', scale_factor=2.0)
    # save as html
    fig.save(FIGURE_DIR / f'Application_areas_{cat}_ts.html')
    

### Applications: detailed breakdowns of major categories

In [ ]:
CATS_COLOURS = {
    'Health': pu.NESTA_COLOURS[0],
    'Development & learning': pu.NESTA_COLOURS[1],
    'Child care & preschool': pu.NESTA_COLOURS[4],
    'Parenting': pu.NESTA_COLOURS[9],
    'Society': pu.NESTA_COLOURS[2],
    'Biosciences': pu.NESTA_COLOURS[5],
}

In [ ]:
applications_trends = (
    pd.concat([
        ukri_application_trends.assign(dataset='Research funding'),
        openalex_application_trends.assign(dataset='Publications'),
        patent_application_trends.assign(dataset='Patents'),
        crunchbase_application_trends.assign(dataset='Venture funding'),
    ], ignore_index=True)
    .assign(amount = lambda df: df.amount / 1000)
)

In [ ]:
# bar chart
def create_application_bar_chart(
        data, 
        dataset_name, 
        title, 
        values, 
        show_yaxis=True, 
        category='type', 
        fig_height=150,
        sort_order = CATS,
    ):
    fig = (
        alt.Chart(
            data.query(f"dataset == '{dataset_name}'"),
            height=fig_height,
            width=100,
        )
        .encode(
            x=alt.X(f'{values}:Q', title=title),
            y=alt.Y(f'{category}:N', title='', sort=sort_order, axis=alt.Axis(labels=False) if not show_yaxis else alt.Axis()),
            color=alt.Color(
                f'type:N', 
                scale=alt.Scale(domain=list(CATS_COLOURS.keys()), range=list(CATS_COLOURS.values())),
                legend=None,
            ),
            tooltip=['dataset', 'type', values, 'growth'],
            text=alt.Text(f'{values}:Q', format=".0f")
        )
        .properties(
            title=dataset_name
        )
    )
    return fig.mark_bar() + fig.mark_text(align='left', dx=2)

def application_bar_charts(applications_trends, category, fig_height=150, sort_order=CATS):
    fig = alt.hconcat(
        create_application_bar_chart(
            applications_trends,
            'Research funding',
            'Research funding (£ millions)',
            'amount',
            show_yaxis=True,
            category=category,
            fig_height=fig_height,
            sort_order=sort_order,
        ),
        create_application_bar_chart(
            applications_trends,
            'Publications',
            'Publication counts',
            'counts',
            show_yaxis=False,
            category=category, 
            fig_height=fig_height,
            sort_order=sort_order,                   
        ),
        create_application_bar_chart(
            applications_trends,
            'Patents',
            'Patent counts',
            'counts',
            show_yaxis=False,
            category=category,  
            fig_height=fig_height,      
            sort_order=sort_order,            
        ),    
        create_application_bar_chart(
            applications_trends,
            'Venture funding',
            'Investment (£ millions)',
            'amount',
            show_yaxis=False,
            category=category,   
            fig_height=fig_height,  
            sort_order=sort_order,               
        )
    ).configure_title(anchor='middle')
    return pu.configure_plots(fig)

fig = application_bar_charts(applications_trends, 'type')
fig

In [ ]:
fig.save(FIGURE_DIR / 'Applications_bar_charts_major.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_bar_charts_major.html')

### Applications: detailed breakdowns of minor categories

In [ ]:
def add_missing_subtypes(trends_minor, topics_df, dataset_name):
    missing_subtypes = (
        set(topics_df.subtype.unique()) - set(trends_minor.subtype.unique())
    )
    return (
        pd.concat([
            trends_minor,
            pd.DataFrame(
                {
                    'subtype': list(missing_subtypes),
                    'amount': 0,
                    'counts': 0,
                    'dataset': dataset_name,
                }
            )
        ])
    )

def process_minor_application_trends_df(trends_minor, dataset_name):
    return (
        trends_minor
        .dropna(subset=['type'])
        .assign(dataset = dataset_name)
        .pipe(add_missing_subtypes, topics_df, dataset_name)
        .drop(columns=['type', 'type_'])
        .merge(topics_df[['subtype', 'type']], on='subtype', how='left')        
    )

In [ ]:
ukri_application_trends_minor = process_minor_application_trends_df(
    get_application_trends(
        ukri_exploded_df.query("year <= 2023"), 
        ukri_tech_ids, 
        ukri_tech_ids_5y, 
        ['amount', 'id'],
        column = 'subtype',
    ),
    dataset_name = 'Research funding'
)

openalex_application_trends_minor = process_minor_application_trends_df(
    get_application_trends(
        openalex_exploded_df.query("year <= 2023"), 
        openalex_tech_ids, 
        openalex_tech_ids_5y, 
        ['id'],
        column = 'subtype',
    ),
    dataset_name = 'Publications'
)

patents_application_trends_minor = process_minor_application_trends_df(
    get_application_trends(
        patents_exploded_df.query("year <= 2023"), 
        patents_tech_ids, 
        patents_tech_ids_5y, 
        ['id'],
        column = 'subtype',
    ),
    dataset_name = 'Patents'
)

crunchbase_application_trends_minor = process_minor_application_trends_df(
    get_application_trends(
        crunchbase_exploded_df.query("year <= 2023"), 
        crunchbase_tech_ids, 
        crunchbase_tech_ids_5y, 
        ['amount', 'id'],
        column = 'subtype',
    ),
    dataset_name = 'Venture funding'
)

applications_trends_minor = (
    pd.concat([
        ukri_application_trends_minor,
        openalex_application_trends_minor,
        patents_application_trends_minor,
        crunchbase_application_trends_minor,
    ], ignore_index=True)
    .assign(amount = lambda df: df.amount / 1000)
)

sort_order_df = (
    applications_trends_minor
    .groupby(['type', 'subtype'])
    .agg(total=('counts', 'sum'))
    .reset_index()
    .assign(type = lambda df: df.type.astype('category').cat.set_categories(CATS))
    .sort_values(['type', 'total'], ascending=[True, False])
    .dropna(subset=['type'])
)

In [ ]:
applications_trends_minor.head(1)

In [ ]:
fig = (
    application_bar_charts(
        (
            applications_trends_minor
            .query("subtype in @sort_order_df.subtype")
            .query("subtype != 'Expressive arts and design'")
        ),
        'subtype',
        fig_height=400,
        sort_order = sort_order_df.subtype.to_list()
    )
    .configure_axisY(grid=True)
    .configure_axisX(grid=False)
)
fig.display()
fig.save(FIGURE_DIR / 'Applications_bar_charts_minor.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_bar_charts_minor.html')

### Applications x Digital tech: detailed breakdowns of major categories


In [ ]:
def applications_x_digital_tech_df_type(
        data_exploded_df, 
        applications_trends, 
        tech_ids,
        tech_ids_5y,
        values
    ):
    # Empty dataframe with all subtypes
    _df = topics_df.query("type in @CATS").query("topic != 'arts'")[['type']].drop_duplicates()
    _values = 'id' if values == 'counts' else values
    # Go through each tech category
    for tech_topic in ['AI', 'Internet', 'Mobile', 'Immersive tech']:
        # Select relevant technology type
        selected_ids = data_exploded_df.query("subtype == @tech_topic").id.to_list()
        # Get application trends for this subset
        counts_df = get_application_trends(
            data_exploded_df.query("year <= 2023").query("id in @selected_ids"), 
            tech_ids, 
            tech_ids_5y, 
            [_values],
            column = 'type',
        )[[values, 'type']].rename(columns={values: tech_topic})
        # Add to the final dataframe
        _df = _df.merge(counts_df, on='type', how='left')
    _df = _df.fillna(0)

    final_df = (
        applications_trends[['type', values]]
        .merge(_df, on='type')
        .rename(columns={values: 'Total'})
        .assign(subtype = lambda df: df.type.astype('category').cat.set_categories(CATS))
        .sort_values('type')
    )[['type', 'AI', 'Mobile', 'Internet', 'Immersive tech', 'Total']]

    return final_df

In [ ]:
ukri_x_digital_tech_major = applications_x_digital_tech_df_type(
    ukri_exploded_df.assign(amount = lambda df: df.amount/1000).query("year <= 2023"), 
    applications_trends.query("dataset == 'Research funding'"), 
    ukri_tech_ids,
    ukri_tech_ids_5y,
    'amount'
)

openalex_x_digital_tech_major = applications_x_digital_tech_df_type(
    openalex_exploded_df.query("year <= 2023"), 
    applications_trends.query("dataset == 'Publications'"), 
    openalex_tech_ids,
    openalex_tech_ids_5y,
    'counts'
)

patents_x_digital_tech_major = applications_x_digital_tech_df_type(
    patents_exploded_df.query("year <= 2023"), 
    applications_trends.query("dataset == 'Patents'"), 
    patents_tech_ids,
    patents_tech_ids_5y,
    'counts'
)

crunchbase_x_digital_tech_major = applications_x_digital_tech_df_type(
    crunchbase_exploded_df.assign(amount = lambda df: df.amount/1000).query("year <= 2023"), 
    applications_trends.query("dataset == 'Venture funding'"), 
    crunchbase_tech_ids,
    crunchbase_tech_ids_5y,
    'amount'
)

ukri_x_digital_tech_major_counts = applications_x_digital_tech_df_type(
    ukri_exploded_df.query("year <= 2023"), 
    applications_trends.query("dataset == 'Research funding'"), 
    ukri_tech_ids,
    ukri_tech_ids_5y,
    'counts'
)

crunchbase_x_digital_tech_major_counts = applications_x_digital_tech_df_type(
    crunchbase_exploded_df.query("year <= 2023"), 
    applications_trends.query("dataset == 'Venture funding'"), 
    crunchbase_tech_ids,
    crunchbase_tech_ids_5y,
    'counts'
)


In [ ]:
def applications_x_digital_tech_chart_major(df, dataset_name, values='amount'):
    # transform to long format
    df_long = df.melt(id_vars=['type'], var_name='column', value_name=values)
    # altair facet grid with one column per digital tech area + total
    fig = (
        alt.Chart(
            width = 120,
        )
        .mark_bar()
        .encode(
            x=alt.X(f'{values}:Q', title=''),
            y=alt.Y('type:N', title='', sort=sort_order_df.subtype.to_list()),        
            # use the scale defined above
            color=alt.Color('type:N', scale=alt.Scale(domain=CATS, range=list(CATS_COLOURS.values())), legend=None),
            tooltip=['type', values]
        )
    )

    text = fig.mark_text(align='left', dx=2).encode(text=alt.Text(f'{values}:Q', format=".0f"))
    # adjust grid
    fig = (
        alt.layer(fig, text, data=df_long)
        .facet(
            column=alt.Column('column:N', title='', sort=['AI', 'Mobile', 'Internet', 'Immersive tech', 'Total'])
        ) 
    )       
    fig = (
        fig
        .configure_axisY(grid=True)
        .configure_axisX(grid=False)
    )
    fig = pu.configure_plots(fig, chart_title = dataset_name)
    return fig

In [ ]:
applications_x_digital_tech_chart_major(
    ukri_x_digital_tech_major, 
    'Research funding (£ millions)',
    'amount'
)

In [ ]:
fig = applications_x_digital_tech_chart_major(
    ukri_x_digital_tech_major, 
    'Research funding (£ millions)',
    'amount'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_ukri.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_ukri.html')

fig = applications_x_digital_tech_chart_major(
    openalex_x_digital_tech_major, 
    'Publications',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_openalex.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_openalex.html')

fig = applications_x_digital_tech_chart_major(
    patents_x_digital_tech_major, 
    'Patents',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_patents.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_patents.html')

fig = applications_x_digital_tech_chart_major(
    crunchbase_x_digital_tech_major, 
    'Venture funding (£ millions)',
    'amount'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_crunchbase.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_crunchbase.html')

fig = applications_x_digital_tech_chart_major(
    ukri_x_digital_tech_major_counts, 
    'Research projects counts',
    'counts'
)

fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_ukri_counts.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_ukri_counts.html')

fig = applications_x_digital_tech_chart_major(
    crunchbase_x_digital_tech_major_counts, 
    'Number of funding rounds',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_crunchbase_counts.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_crunchbase_counts.html')



### Applications x Digital tech: detailed breakdowns of minor categories

In [ ]:
def applications_x_digital_tech_df(
        data_exploded_df, 
        application_trends_minor, 
        tech_ids,
        tech_ids_5y,
        values
    ):
    # Empty dataframe with all subtypes
    _df = topics_df.query("type in @CATS").query("topic != 'arts'")[['subtype']]
    _values = 'id' if values == 'counts' else values
    # Go through each tech category
    for tech_topic in ['AI', 'Internet', 'Mobile', 'Immersive tech']:
        # Select relevant technology type
        selected_ids = data_exploded_df.query("subtype == @tech_topic").id.to_list()
        # Get application trends for this subset
        counts_df = get_application_trends(
            data_exploded_df.query("year <= 2023").query("id in @selected_ids"), 
            tech_ids, 
            tech_ids_5y, 
            [_values],
            column = 'subtype',
        )[[values, 'subtype']].rename(columns={values: tech_topic})
        # Add to the final dataframe
        _df = _df.merge(counts_df, on='subtype', how='left')
    _df = _df.fillna(0)

    final_df = (
        application_trends_minor[['subtype', values]]
        .merge(_df, on='subtype')
        .rename(columns={values: 'Total'})
        .merge(topics_df[['subtype', 'type']], on='subtype', how='left')
        .assign(subtype = lambda df: df.subtype.astype('category').cat.set_categories(sort_order_df.subtype.to_list()))
        .sort_values('subtype')
    )[['type', 'subtype', 'AI', 'Mobile', 'Internet', 'Immersive tech', 'Total']]

    return final_df

In [ ]:
ukri_x_digital_tech = applications_x_digital_tech_df(
    ukri_exploded_df.assign(amount = lambda df: df.amount/1000).query("year <= 2023"), 
    ukri_application_trends_minor.assign(amount = lambda df: df.amount/1000), 
    ukri_tech_ids,
    ukri_tech_ids_5y,
    'amount'
)

crunchbase_x_digital_tech = applications_x_digital_tech_df(
    crunchbase_exploded_df.assign(amount = lambda df: df.amount/1000).query("year <= 2023"), 
    crunchbase_application_trends_minor.assign(amount = lambda df: df.amount/1000), 
    crunchbase_tech_ids,
    crunchbase_tech_ids_5y,
    'amount'
)

patents_x_digital_tech = applications_x_digital_tech_df(
    patents_exploded_df.query("year <= 2023"), 
    patents_application_trends_minor, 
    patents_tech_ids,
    patents_tech_ids_5y,
    'counts'
)

openalex_x_digital_tech = applications_x_digital_tech_df(
    openalex_exploded_df.query("year <= 2023"), 
    openalex_application_trends_minor, 
    openalex_tech_ids,
    openalex_tech_ids_5y,
    'counts'
)

ukri_x_digital_tech.to_csv(FIGURE_DIR / 'Applications_x_digital_tech_ukri.csv', index=False)
crunchbase_x_digital_tech.to_csv(FIGURE_DIR / 'Applications_x_digital_tech_crunchbase.csv', index=False)
patents_x_digital_tech.to_csv(FIGURE_DIR / 'Applications_x_digital_tech_patents.csv', index=False)
openalex_x_digital_tech.to_csv(FIGURE_DIR / 'Applications_x_digital_tech_openalex.csv', index=False)

In [ ]:
def applications_x_digital_tech_chart(df, dataset_name, values='amount'):
    # transform to long format
    df_long = df.melt(id_vars=['type', 'subtype'], var_name='column', value_name=values)
    # altair facet grid with one column per digital tech area + total
    fig = (
        alt.Chart(
            width = 120,
        )
        .mark_bar()
        .encode(
            x=alt.X(f'{values}:Q', title=''),
            y=alt.Y('subtype:N', title='', sort=sort_order_df.subtype.to_list()),        
            # use the scale defined above
            color=alt.Color('type:N', scale=alt.Scale(domain=CATS, range=list(CATS_COLOURS.values())), legend=None),
            tooltip=['subtype', values]
        )
    )

    text = fig.mark_text(align='left', dx=2).encode(text=alt.Text(f'{values}:Q', format=".0f"))
    # adjust grid
    fig = (
        alt.layer(fig, text, data=df_long)
        .facet(
            column=alt.Column('column:N', title='', sort=['AI', 'Mobile', 'Internet', 'Immersive tech', 'Total'])
        ) 
    )       
    fig = (
        fig
        .configure_axisY(grid=True)
        .configure_axisX(grid=False)
    )
    fig = pu.configure_plots(fig, chart_title = dataset_name)
    return fig

In [ ]:
applications_x_digital_tech_chart(
    ukri_x_digital_tech, 
    'Research funding (£ millions)',
    'amount'
)

In [ ]:
fig = applications_x_digital_tech_chart(
    ukri_x_digital_tech, 
    'Research funding (£ millions)',
    'amount'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_ukri.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_ukri.html')

fig = applications_x_digital_tech_chart(
    crunchbase_x_digital_tech, 
    'Venture funding (£ millions)',
    'amount'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_crunchbase.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_crunchbase.html')

fig = applications_x_digital_tech_chart(
    patents_x_digital_tech, 
    'Patent counts',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_patents.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_patents.html')

fig = applications_x_digital_tech_chart(
    openalex_x_digital_tech, 
    'Publication counts',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_openalex.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_openalex.html')



In [ ]:
ukri_x_digital_tech_counts = applications_x_digital_tech_df(
    ukri_exploded_df.query("year <= 2023"), 
    ukri_application_trends_minor, 
    ukri_tech_ids,
    ukri_tech_ids_5y,
    'counts'
)

fig = applications_x_digital_tech_chart(
    ukri_x_digital_tech_counts, 
    'Research projects counts',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_ukri_counts.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_ukri_counts.html')



In [ ]:
applications_x_digital_tech_chart(
    ukri_x_digital_tech_counts, 
    'Research projects counts',
    'counts'
)

In [ ]:
crunchbase_x_digital_tech_counts = applications_x_digital_tech_df(
    crunchbase_exploded_df.query("year <= 2023"), 
    crunchbase_application_trends_minor, 
    crunchbase_tech_ids,
    crunchbase_tech_ids_5y,
    'counts'
)

fig = applications_x_digital_tech_chart(
    crunchbase_x_digital_tech_counts, 
    'Number of funding rounds',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_crunchbase_counts.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_crunchbase_counts.html')


## Growth and magnitude trends diagrams

In [ ]:
chart_trends._epsilon = 0.05

In [ ]:
ukri_trends_typology = chart_trends.estimate_trend_type(
    ukri_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]
ukri_trends_typology

In [ ]:
chart_trends._epsilon = 0.025
mid_point = ukri_trends_typology.magnitude.median()

fig = chart_trends.mangitude_vs_growth_chart(
    data = (
        ukri_trends_typology
        .rename(columns = {'type': 'category'})
        .assign(growth = lambda df: df.growth / 100)
        # .assign(magnitude = lambda df: df.magnitude / 1000)
    ),
    x_limit=10000,
    y_limit= 3.5,
    mid_point=mid_point,
    baseline_growth=0,
    text_column = "category",
    values_label = "Average new funding per year (£ thousands)",
)
fig.display()
fig.save(FIGURE_DIR / 'Figure7_ukri_trends.png', scale_factor=2.0)

In [ ]:
openalex_trends_typology = chart_trends.estimate_trend_type(
    openalex_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]
openalex_trends_typology

In [ ]:
chart_trends._epsilon = 0.075
mid_point = openalex_trends_typology.magnitude.median()

fig = chart_trends.mangitude_vs_growth_chart(
    data = (
        openalex_trends_typology
        .rename(columns = {'type': 'category'})
        .assign(growth = lambda df: df.growth / 100)
    ),
    x_limit=300,
    y_limit= 1.2,
    mid_point=mid_point,
    baseline_growth=0,
    text_column = "category",
    values_label = "Average number of new publications per year",
)
fig.display()
fig.save(FIGURE_DIR / 'Figure7_openalex_trends.png', scale_factor=2.0)

In [ ]:
patent_trends_typology = chart_trends.estimate_trend_type(
    patent_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]
patent_trends_typology

In [ ]:
chart_trends._epsilon = 0.075
mid_point = patent_trends_typology.magnitude.median()

fig = chart_trends.mangitude_vs_growth_chart(
    data = (
        patent_trends_typology
        .rename(columns = {'type': 'category'})
        .assign(growth = lambda df: df.growth / 100)
    ),
    x_limit=300,
    y_limit= 2,
    mid_point=mid_point,
    baseline_growth=0,
    text_column = "category",
    values_label = "Average number of new patent applications per year",
)
fig.display()
fig.save(FIGURE_DIR / 'Figure7_patent_trends.png', scale_factor=2.0)

In [ ]:
crunchbase_trends_typology = chart_trends.estimate_trend_type(
    crunchbase_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]
crunchbase_trends_typology

In [ ]:
chart_trends._epsilon = 0.025
mid_point = crunchbase_trends_typology.magnitude.median()

fig = chart_trends.mangitude_vs_growth_chart(
    data = (
        crunchbase_trends_typology
        .rename(columns = {'type': 'category'})
        .assign(growth = lambda df: df.growth / 100)
    ),
    x_limit=600000,
    y_limit= 1,
    mid_point=mid_point,
    baseline_growth=0,
    text_column = "category",
    values_label = "Average new venture funding per year (£ thousands)",
)
fig.display()
fig.save(FIGURE_DIR / 'Figure7_patent_trends.png', scale_factor=2.0)